# Kaggle CPU runner

No GPU, secrets or paid tier is required. This notebook has not been executed in the hosted environment. Dataset terms apply; do not publish third-party source assets.


## Runtime and disk

In [ ]:
import os, sys, shutil, platform
from pathlib import Path
print(sys.version, platform.platform())
print("Disk free:", shutil.disk_usage(Path.cwd()).free)
print("CPU count:", os.cpu_count(), "GPU is not needed")


## Obtain project source

In [ ]:
# Upload the project or set a repository URL you own.
PROJECT_REPOSITORY = ""  # Supply your actual repository URL if cloning.
if PROJECT_REPOSITORY:
    import subprocess
    subprocess.run(["git", "clone", PROJECT_REPOSITORY, "assignment3-cv"], check=True)
    os.chdir("assignment3-cv")
if not Path("pyproject.toml").is_file():
    raise RuntimeError("Upload the project and change directory, or supply PROJECT_REPOSITORY")


## Install prerequisites

If Internet is disabled, install from authorized attached wheels and use offline source archives. Read-only inputs belong under `/kaggle/input/<dataset-slug>/...`, with the actual slug supplied by the user.

In [ ]:
import shutil
if not shutil.which("tesseract"):
    print("Install tesseract-ocr and tesseract-ocr-eng only if the environment permits")
%pip install -c requirements-lock.txt -e ".[dev,report]"
RESULTS_DIR = "/kaggle/working/assignment3-results"


## Configuration and offline checks

In [ ]:
import yaml
Path("runner.yaml").write_text(yaml.safe_dump({"results_dir":RESULTS_DIR, "data_dir":"data"}))
!python -m cvdoc doctor --config runner.yaml
!python -m pytest -q


## User-controlled dataset preparation

In [ ]:
# Personal review of https://guillaumejaume.github.io/FUNSD/work/ is required.
ACCEPT_FUNSD_TERMS_AND_CONFIRM_AGE_18 = False
DOWNLOAD_SMARTDOC = False
FUNSD_SOURCE = ""  # Existing authorized ZIP, e.g. a read-only Kaggle input path.
SMARTDOC_SOURCE = ""  # Existing public frames.tar.gz; no invented dataset slug.
import subprocess
base = [sys.executable, "-m", "cvdoc"]
if ACCEPT_FUNSD_TERMS_AND_CONFIRM_AGE_18:
    args = ["data", "prepare-funsd", "--accept-terms", "--config", "runner.yaml"]
    args += ["--source", FUNSD_SOURCE] if FUNSD_SOURCE else ["--download"]
    subprocess.run(base + args, check=True)
else:
    print("FUNSD preparation pending personal acceptance")
if SMARTDOC_SOURCE or DOWNLOAD_SMARTDOC:
    args = ["data", "prepare-smartdoc", "--config", "runner.yaml"]
    args += ["--source", SMARTDOC_SOURCE] if SMARTDOC_SOURCE else ["--download"]
    subprocess.run(base + args, check=True)
else:
    print("SmartDoc preparation pending source/download selection")


## Deterministic manifests

In [ ]:
# Run after preparing at least one dataset.
subprocess.run(base + ["data", "make-splits", "--seed", "42", "--config", "runner.yaml"], check=True)


## Validation smoke experiments

In [ ]:
# Explicitly select the available track(s). These are validation smoke runs.
RUN_SMARTDOC = False
RUN_FUNSD = False
if RUN_SMARTDOC:
    for method in ("baseline", "refined"):
        subprocess.run(base + ["benchmark", "geometry", "--split", "val", "--method", method,
                               "--max-items", "12", "--config", "runner.yaml"], check=True)
if RUN_FUNSD:
    subprocess.run(base + ["benchmark", "ocr", "--split", "val", "--variants", "frontal,mild,strong",
                           "--max-items", "10", "--config", "runner.yaml"], check=True)


## Frozen final benchmark

In [ ]:
# Freeze using completed validation run paths before final test execution.
RUN_FINAL_TEST = False
FROZEN_CONFIG = "configs/frozen.yaml"
if RUN_FINAL_TEST:
    if not Path(FROZEN_CONFIG).is_file():
        raise RuntimeError("Freeze the validated config with scripts/freeze_config.py first")
    subprocess.run(base + ["benchmark", "all", "--split", "test", "--config", FROZEN_CONFIG], check=True)


## Display real aggregate figures

In [ ]:
from IPython.display import display, Image
for figure in sorted(Path(RESULTS_DIR).glob("*/figures/*.png")):
    print(figure)
    display(Image(filename=str(figure)))


## Export small results

In [ ]:
# Export aggregate metrics and own figures only, never source images or transcripts.
RUN_DIRECTORY = ""  # Actual printed run directory.
EXPORT_PATH = ""  # User-defined ZIP path, optionally inside explicitly mounted Drive.
if RUN_DIRECTORY and EXPORT_PATH:
    subprocess.run([sys.executable, "scripts/export_metrics.py", RUN_DIRECTORY, EXPORT_PATH], check=True)
